In [9]:
%load_ext autoreload
%autoreload 2

import scanpy as sc
import pandas as pd
import decoupler as dc
import matplotlib as mpl
import matplotlib.pyplot as plt

import sys
import os
from pathlib import Path

# Repo-relative paths: inputs from imports_stable/, outputs to analysis_outs/<module>/
_here = Path.cwd().resolve()
REPO_DIR = str(next(p for p in [_here, *_here.parents] if (p / "imports_stable").is_dir()))
IMPORTS_DIR = f"{REPO_DIR}/imports_stable"
OUTS_DIR = f"{REPO_DIR}/analysis_outs/05_in_vitro_differentiation_RNAseq_SIG18"
sys.path.insert(0, f"{REPO_DIR}/functions")
import scanpy_custom as scc

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
# build long-format loadings from the sPCA components (same as the export in 03)
spca_components = pd.read_csv(IMPORTS_DIR + "/SIG18/analysis_outs/spca/zscore_degs/zscore_degs_unscaled_alpha10.0_sPCA_components.csv")
spca_components.columns = ['gene'] + ['comp_' + c for c in spca_components.columns[1:]]

# Component Selection

In [11]:
# format for decoupler
net = (spca_components
       .melt(id_vars='gene', var_name='source', value_name='weight')
       .rename(columns={'gene':'target'})
       .query('weight != 0')
       .sort_values(['source', 'weight'], ascending=[True, False])
       .reset_index(drop=True))

# define genes used in scoring
comp_genes = net['target'].unique().tolist()

# SIG18 Scoring

Here, I will calculate scores for the sPCA components using a weighted mean of scaled log1p_norm component genes.

In [12]:
counts = pd.read_csv(IMPORTS_DIR + '/SIG18/processing_outs/count_matrix_umiDeDup_SIG18.csv', index_col=0)
feature_names = pd.read_csv(IMPORTS_DIR + '/SIG18/processing_outs/featureNames_SIG18.csv', index_col=0)

In [14]:
# make filtered counts matrix
feature_names_filtered = feature_names[feature_names['gene'].isin(comp_genes)]
counts_filtered = counts.loc[counts.index.isin(feature_names_filtered.index),:]
counts_filtered = counts_filtered.merge(feature_names_filtered, left_index=True, right_index=True)
counts_filtered = counts_filtered.set_index('gene')
counts_filtered.drop(columns=['category'], inplace=True)

# convert counts into anndata object and preprocess
rna = sc.AnnData(counts_filtered.T)
rna.var_names_make_unique()
sc.pp.normalize_total(rna)
sc.pp.log1p(rna)
sc.pp.scale(rna)

# build out obs dataframe
obs_df = pd.DataFrame({"sample": counts_filtered.columns})
obs_df[['ligand1','ligand2','mouse','sex','well','column','project']] = obs_df['sample'].str.split('_', expand=True)
obs_df = obs_df.assign(condition=lambda x: x['ligand1'] + '_' + x['ligand2'])
rna.obs = rna.obs.merge(obs_df, left_index=True, right_on='sample')

# average across treatment conditions
rna_pb = sc.get.aggregate(rna, by=['condition','mouse','sex','well'], func='mean')

/home/wange7/miniforge3/envs/scanpy_standard/lib/python3.12/site-packages/anndata/_core/anndata.py:1793: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/wange7/miniforge3/envs/scanpy_standard/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


In [15]:
dc.mt.waggr(rna_pb, net, tmin=5, layer='mean', verbose=True, times=0)

2026-01-18 14:42:07 | [INFO] waggr - Running waggr
2026-01-18 14:42:07 | [INFO] Extracted omics mat with 96 rows (observations) and 3210 columns (features)
2026-01-18 14:42:07 | [INFO] Network adjacency matrix has 3205 unique features and 57 unique sources
2026-01-18 14:42:07 | [INFO] waggr - using function _wmean
2026-01-18 14:42:07 | [INFO] waggr - calculating scores for 57 sources across 96 observations
2026-01-18 14:42:07 | [INFO] waggr - adjusting p-values by FDR
2026-01-18 14:42:07 | [INFO] waggr - done


In [16]:
# Extract score_waggr and padj_waggr from rna_pb.obsm
score_df = pd.DataFrame(rna_pb.obsm['score_waggr'], 
                        index=rna_pb.obs_names, 
                        columns=rna_pb.obsm['score_waggr'].columns)
score_df = score_df.rename_axis('interaction')

In [ ]:
os.makedirs(f"{OUTS_DIR}/spca/zscore_degs", exist_ok=True)
score_df.to_csv(OUTS_DIR + "/spca/zscore_degs/zscore_degs_alpha10.0_waggr_score.csv")